In [1]:
%env HF_ENDPOINT=https://hf-mirror.com

env: HF_ENDPOINT=https://hf-mirror.com


In [1]:
# 加载模型与TOkenizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

model_name = "Qwen/Qwen3-4B"
model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 398/398 [01:17<00:00,  5.11it/s]


In [2]:
model.device

device(type='cpu')

In [2]:
# 数据集 处理数据集至openai格式
from datasets import load_dataset
dataset_dict = load_dataset("json",data_files={"train":"data/keywords_data_train.jsonl",
                                              "test":"data/keywords_data_test.jsonl"})
# 转成openai格式
def map_func(exapmle):
    conversation = exapmle["conversation"]
    messages=[]
    for item in conversation:
        messages.append({"role":"user","content":item["human"]})
        messages.append({"role":"assistant","content":item["assistant"]})
    return {"messages":messages}

dataset_dict=dataset_dict.map(map_func,batched=False,remove_columns=["conversation_id","category","conversation","dataset"])

In [3]:
from peft import LoraConfig
from trl import SFTConfig,SFTTrainer
# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 4
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

training_args = SFTConfig(
    output_dir="/home/tianjp/llmLearn/stf/Qwen3-4B/sft-full",
    max_steps=1000,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_total_limit=2,
    save_steps=100,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50,
    assistant_only_loss=True,
)


peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    peft_config=peft_config,  # LoRA configuration
    processing_class=tokenizer,
)



In [5]:
dataloader = trainer.get_train_dataloader()
batch=next(iter(dataloader))
batch["input_ids"].shape

torch.Size([2, 166])

In [6]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,0.837095,1.183866
200,0.807688,1.118241
300,1.020633,1.097234
400,1.089887,1.090846
500,0.805879,1.072497
600,1.176439,1.066406
700,0.797194,1.064042
800,0.985528,1.059337
900,0.973830,1.055651
1000,1.248136,1.053638


/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.

TrainOutput(global_step=1000, training_loss=1.2580921053886414, metrics={'train_runtime': 1251.9657, 'train_samples_per_second': 1.597, 'train_steps_per_second': 0.799, 'total_flos': 9609682776545280.0, 'train_loss': 1.2580921053886414})

In [7]:
trainer.save_model("/home/tianjp/llmLearn/stf/Qwen3-4B/sft-full/best")

/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/other.py:1419: UserWarning: Unable to fetch remote file due to the following error [Errno 101] Network is unreachable - silently ignoring the lookup for the file config.json in Qwen/Qwen3-4B.
  warnings.warn(
/home/tianjp/anaconda3/envs/llm/lib/python3.12/site-packages/peft/utils/save_and_load.py:372: UserWarning: Could not find a config file in Qwen/Qwen3-4B - will assume that the vocabulary was not modified.
  warnings.warn(
